# 09 · Exportar las capas de las figuras del artículo

No entrena nada. Lee las salidas del notebook 08 v2 (en `/kaggle/working` o adjuntas como *Input*) y produce `paper_figs.zip` con:

- `img/fig3_*.png`: paneles de la figura combinada (verdad, IMAP, UMAP, oráculo DMAP, DMAP+AE, DMAP+AE+jac, AE sin DMAP), **solo los puntos**, fondo transparente;
- `img/rings_*.png`: anillos (oráculo) y latente alineado por semejanza, muestreo aleatorio y cuadrícula;
- `data/readout_vs_d.dat`: error relativo del readout afín frente a d para IMAP, UMAP y DMAP (como la figura 2 de la referencia);
- `data/fig3_meta.json`: réplica usada, número de puntos fuera de rango y el factor β de la cuadrícula.

Copia el contenido de `paper_figs.zip` en las carpetas `img/` y `data/` del proyecto LaTeX y vuelve a compilar: los marcos, etiquetas y colores los pone pgfplots (noche o día según `dacnotes`).

Necesita **Internet: On** para instalar `umap-learn`. Duración: ~5–10 min (UMAP domina).

## Preparación

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
ROOT = Path('/kaggle/working/dmap_vae_kaggle')
INPUT = Path('/kaggle/input')
CU = 'flash_diffusion.cu'
CODE = ['spectral.py', 'vae.py', 'data.py', 'experiment.py', 'launch.py', 'plots.py',
        'checks.py', 'prepare_alanine.py', 'generate_angles.py', 'SOURCE.json',
        'geometry.py', 'beta_sweep.py', 'protocol.py', 'config_beta_sweep.json',
        'config_beta_sweep_torus.json', 'config_beta_sweep_mnist.json', 'config_beta_sweep_alanine.json', 'latent_figure.py', 'export_figs.py']

# 1) Refresh the code on EVERY run (a partial copy left in /kaggle/working is never trusted).
#    The results in /kaggle/working/results_* are not touched.
sources = sorted({q.parent for q in INPUT.rglob('launch.py') if (q.parent / 'spectral.py').exists()})
archives = sorted(INPUT.rglob('dmap_vae_kaggle*.zip'))
if len(sources) > 1:
    raise FileNotFoundError(f'Hay varias copias del paquete en /kaggle/input: {sources}. Deja solo una.')
ROOT.mkdir(parents=True, exist_ok=True)
if sources:
    shutil.copytree(sources[0], ROOT, dirs_exist_ok=True)
elif len(archives) == 1:
    with zipfile.ZipFile(archives[0]) as z:
        z.extractall(ROOT.parent)
else:
    raise FileNotFoundError('Añade el paquete dmap_vae_kaggle como Kaggle Dataset; debe haber una sola copia.')
missing_code = [f for f in CODE if not (ROOT / f).exists()]
assert not missing_code, (f'Faltan archivos del paquete: {missing_code}. Este notebook necesita el '
                          'paquete v5 (dmap_vae_kaggle_v5.zip): sustituye el Dataset anterior.')

# 2) The CUDA source. Kaggle drops or flattens subfolders when a Dataset is built from loose
#    files, so look for it in vendor/, flattened at the root, or inside any attached ZIP.
target = ROOT / 'vendor' / CU
target.parent.mkdir(exist_ok=True)
if not target.exists():
    found = sorted(INPUT.rglob(CU))
    if found:
        shutil.copy2(found[0], target)
    else:
        for arc in sorted(INPUT.rglob('*.zip')):
            with zipfile.ZipFile(arc) as z:
                names = [n for n in z.namelist() if n.endswith('/' + CU) or n == CU]
                if names:
                    target.write_bytes(z.read(names[0]))
                    break
    lic = sorted(INPUT.rglob('LICENSE.FlashDiffusion'))
    if lic and not (target.parent / lic[0].name).exists():
        shutil.copy2(lic[0], target.parent / lic[0].name)
if not target.exists():
    raise FileNotFoundError(
        f'No se encontró {CU} en /kaggle/input. Sube la carpeta vendor/ (o el ZIP completo) '
        'al Kaggle Dataset; al subir archivos sueltos Kaggle omite las subcarpetas.')
expected = json.loads((ROOT / 'SOURCE.json').read_text())['sha256']
actual = hashlib.sha256(target.read_bytes()).hexdigest()
assert actual == expected, f'{CU} no coincide con SOURCE.json ({actual} != {expected}).'
print('Fuente CUDA verificada:', target)

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
import importlib.util
missing = [m for m in ['numpy','scipy','pandas','matplotlib','ninja']
           if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
import torch
# este notebook no necesita GPU

(ROOT / 'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))


## Resultados del 08 v2

In [ ]:
import subprocess, sys, json, shutil
from pathlib import Path
import importlib.util
if importlib.util.find_spec('umap') is None:
    if subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'umap-learn']).returncode:
        print('ATENCIÓN: no se pudo instalar umap-learn (¿Internet off?): la fila UMAP quedará vacía')

def buscar(nombre):
    for base in [Path('/kaggle/working')] + [p.parent for p in sorted(Path('/kaggle/input').rglob(nombre))]:
        f = base / nombre
        if (f / 'metrics.csv').exists():
            return f
    return None
F = {t: buscar(f'results_swiss_final_{t}') for t in
     ('AE_l0', 'AE_l0.1', 'PLAIN_AE_l0', 'GRID_AE_l0', 'GRID_AE_l0.1', 'GRID_PLAIN_AE_l0')}
for k, v in F.items():
    print(f'{k:18s}', v or 'NO ENCONTRADO')
assert F['AE_l0.1'] is not None, 'Falta results_swiss_final_AE_l0.1: adjunta la salida del notebook 08 v2.'
sel = sorted(Path('/kaggle/input').rglob('beta_sweep_swiss_grid/beta_selection.json')) + \
      sorted(Path('/kaggle/working').glob('beta_sweep_swiss_grid/beta_selection.json'))
FACTOR_GRID = json.loads(sel[0].read_text())['selected_factor'] if sel else None
print('factor beta de la cuadrícula =', FACTOR_GRID, '(cópialo en \\gridfactor de main.tex)')

## Exportar

In [ ]:
OUT = Path('/kaggle/working/paper_figs')
args = [sys.executable, str(ROOT / 'export_figs.py'), '--random', str(F['AE_l0.1']), '--out', str(OUT), '--readout',
        '--methods', 'imap', 'umap']
for row, tag, model in (('ae', 'AE_l0', 'ae_warm'), ('aejac', 'AE_l0.1', 'ae_warm'), ('plainae', 'PLAIN_AE_l0', 'plainae')):
    if F[tag] is not None:
        args += ['--rows', row, str(F[tag]), model, '--rings-random', row, str(F[tag]), model]
for row, tag, model in (('ae', 'GRID_AE_l0', 'ae_warm'), ('aejac', 'GRID_AE_l0.1', 'ae_warm'), ('plainae', 'GRID_PLAIN_AE_l0', 'plainae')):
    if F[tag] is not None:
        args += ['--rings-grid', row, str(F[tag]), model]
r = subprocess.run(args, capture_output=True, text=True, cwd=ROOT)
print(r.stdout[-3000:]); print(r.stderr[-3000:])
assert r.returncode == 0, 'falló la exportación'
meta = json.loads((OUT / 'data' / 'fig3_meta.json').read_text())
meta['factor_grid'] = FACTOR_GRID
(OUT / 'data' / 'fig3_meta.json').write_text(json.dumps(meta, indent=1))
shutil.make_archive('/kaggle/working/paper_figs', 'zip', OUT)
print('listo: /kaggle/working/paper_figs.zip', sorted(p.name for p in (OUT / 'img').glob('*.png'))[:8], '...')